In [1]:
import re
import pandas as pd
from tqdm import tqdm

# ============================================================
# CONFIG
# ============================================================
def run_pattern_extraction(file):
    INPUT_CSV = f"data/mindset/{file}.csv"
    OUTPUT_CSV = f"data/demographics/{file}.csv"

    # ============================================================
    # PATTERNS
    # ============================================================

    # Self-reference markers that should immediately precede an unbracketed age+gender token.
    # Matches: "i'm", "i am", "im", "i m", "me", "hi", "hello", "hey".
    _SELF_REF = (
        r"(?:i\s*['‘’ʼ´`]?\s*m"
        r"|i\s+am"
        r"|im(?=\s+\d)"
        r"|me(?=\s+\d)"
        r"|hi[,!]?"
        r"|hello[,!]?"
        r"|hey[,!]?)"
    )

    # Self-disclosure markers that should immediately follow an unbracketed age+gender token.
    # Matches: "here", "and i'm/am/'ve", "looking for", "seeking", "single", "married",
    # "asking for", "need advice/help", "new to", "from <place>".
    _SELF_DISCLOSE = (
        r"(?:here\b"
        r"|and\s+i(?:'?(?:m|ve|am)|\s+am)\b"
        r"|looking\s+for\b"
        r"|seeking\b"
        r"|single\b"
        r"|married\b"
        r"|asking\s+for\b"
        r"|need(?:ing)?\s+(?:some\s+)?(?:advice|help)\b"
        r"|new\s+to\b)"
    )

    AGE_GENDER_PATTERNS = [
        # 1) Bracketed/parenthesized: (34F), [28m], (23/f), (23-F)
        re.compile(
            r"[\(\[]\s*(\d{1,3})\s*[\/\-,]?\s*"
            r"(m|f|nb|male|female|nonbinary|enby)\s*[\)\]]",
            re.I,
        ),
        # 2) Self-referenced: "i'm 24m", "i am 32f", "im 28m", "me 24m", "hi, 32f"
        re.compile(
            rf"\b{_SELF_REF}\s+(?:a\s+|an\s+)?"
            r"(\d{1,3})\s*[\/\-,]?\s*"
            r"(m|f|nb|male|female|nonbinary|enby)\b",
            re.I,
        ),
        # 3) Followed by self-disclosure: "32F here", "28m, looking for", "24m single"
        re.compile(
            r"(?<![.\d$£€¥])\b(\d{1,3})\s*[\/\-,]?\s*"
            r"(m|f|nb|male|female|nonbinary|enby)\b"
            rf"\s*[,!.]?\s+{_SELF_DISCLOSE}",
            re.I,
        ),
    ]

    AGE_PATTERNS = [
        re.compile(r'\b(\d{1,3})\s*(yo|y/o|years?\s*old|yrs?\s*old)\b', re.I),
        re.compile(r'at my age\s*[\(\[]?\s*(\d{1,3})\s*[\)\]]?', re.I),
        re.compile(r'\b(?:i am|i\'m|im)\s+(?:actually\s+)?(\d{1,3})\b', re.I),
        re.compile(r'\b(\d{1,3})\s*-\s*(\d{1,3})\s*year old\b', re.I),
        re.compile(r'\b(\d{1,3})\s*-\s*(\d{1,3})\s*years old\b', re.I),
    ]

    GENDER_PATTERNS = [
        re.compile(r'\b(?:i am|i\'m|im)\s+(?:a\s+)?(girl|boy|man|woman|male|female|nonbinary|nb|enby)\b', re.I),
        re.compile(r'\bbiologically\s+(male|female)\b', re.I),
        re.compile(r'\b(?:a\s+)?(detrans woman|detrans man|trans woman|trans man)\b', re.I),
    ]

    EDUCATION_PATTERNS = [
        re.compile(r'\b(high school|secondary school)\b', re.I),
        re.compile(r"\b(college student|undergrad|undergraduate|bachelor['']?s?|b\.a\.|b\.s\.)\b", re.I),
        re.compile(r"\b(master['']?s?|m\.s\.|m\.a\.)\b", re.I),
        re.compile(r'\b(phd|ph\.d\.|doctorate|doctoral student|grad student|graduate student)\b', re.I),
        re.compile(r'\b(professor|postdoc|researcher)\b', re.I),
        re.compile(r'\b(student|freshman|sophomore|junior|senior)\b', re.I),
    ]

    # ============================================================
    # NORMALIZATION
    # ============================================================

    def normalize_gender(value: str):
        if not value:
            return None
        v = value.strip().lower()
        mapping = {
            "m": "male", "male": "male", "man": "male", "boy": "male",
            "f": "female", "female": "female", "woman": "female", "girl": "female",
            "nb": "nonbinary", "nonbinary": "nonbinary", "enby": "nonbinary",
            "trans man": "trans_man", "trans woman": "trans_woman",
            "detrans man": "detrans_man", "detrans woman": "detrans_woman",
        }
        return mapping.get(v, v)

    def normalize_education(value: str):
        if not value:
            return None
        v = value.strip().lower()
        if v in {"high school", "secondary school"}:
            return "high_school"
        if v in {"college student", "undergrad", "undergraduate", "bachelor's", "bachelors", "b.a.", "b.s."}:
            return "undergraduate"
        if v in {"master's", "masters", "m.s.", "m.a."}:
            return "masters"
        if v in {"phd", "ph.d.", "doctorate", "doctoral student", "grad student", "graduate student"}:
            return "phd_or_grad"
        if v in {"professor", "postdoc", "researcher"}:
            return "academic_staff"
        if v in {"student", "freshman", "sophomore", "junior", "senior"}:
            return "student"
        return v

    def is_valid_age(age):
        try:
            return 10 <= int(age) <= 100
        except:
            return False

    # ============================================================
    # SINGLE-POST EXTRACTION (unchanged logic)
    # ============================================================

    def extract_demographics(text):
        if pd.isna(text):
            text = ""
        text = str(text)

        # Strip quoted / reported speech
        text = re.sub(r'"[^"]*"', ' ', text)
        text = re.sub(r'“[^”]*”', ' ', text)
        text = re.sub(r'(?m)^[ \t]*>.*$', ' ', text)

        result = {
            "age": None, "age_min": None, "age_max": None,
            "gender": None, "education_level": None,
            "matched_excerpt": None, "pattern_type": None, "confidence": "low",
        }

        # 1. Age + gender together
        for pat in AGE_GENDER_PATTERNS:
            m = pat.search(text)
            if m and is_valid_age(m.group(1)):
                result.update(age=int(m.group(1)), gender=normalize_gender(m.group(2)),
                            matched_excerpt=m.group(0), pattern_type="age_gender", confidence="high")
                return result

        # 2. Age range
        for pat in AGE_PATTERNS[-2:]:
            m = pat.search(text)
            if m and is_valid_age(m.group(1)) and is_valid_age(m.group(2)):
                result.update(age_min=int(m.group(1)), age_max=int(m.group(2)),
                            matched_excerpt=m.group(0), pattern_type="age_range", confidence="medium")
                for gpat in GENDER_PATTERNS:
                    gm = gpat.search(text)
                    if gm:
                        result["gender"] = normalize_gender(gm.group(1))
                        break
                for epat in EDUCATION_PATTERNS:
                    em = epat.search(text)
                    if em:
                        result["education_level"] = normalize_education(em.group(1))
                        break
                return result

        # 3. Age only
        for pat in AGE_PATTERNS[:-2]:
            m = pat.search(text)
            if m and is_valid_age(m.group(1)):
                result.update(age=int(m.group(1)), matched_excerpt=m.group(0),
                            pattern_type="age_only", confidence="medium")
                break

        # 4. Gender only
        for pat in GENDER_PATTERNS:
            m = pat.search(text)
            if m:
                result["gender"] = normalize_gender(m.group(1))
                if result["matched_excerpt"] is None:
                    result.update(matched_excerpt=m.group(0), pattern_type="gender_only")
                result["confidence"] = "medium" if result["age"] is None else result["confidence"]
                break

        # 5. Education only
        for pat in EDUCATION_PATTERNS:
            m = pat.search(text)
            if m:
                result["education_level"] = normalize_education(m.group(1))
                if result["matched_excerpt"] is None:
                    result.update(matched_excerpt=m.group(0), pattern_type="education_only")
                if result["age"] is None and result["gender"] is None:
                    result["confidence"] = "medium"
                break

        return result

    # ============================================================
    # USER-LEVEL EXTRACTION: merge across all posts for a user
    # ============================================================

    def extract_user_demographics(texts):
        """Run extraction on every post for one user; merge the best result."""
        best = {"age": None, "age_min": None, "age_max": None,
                "gender": None, "education_level": None,
                "matched_excerpt": None, "pattern_type": None, "confidence": "low"}
        conf_rank = {"low": 0, "medium": 1, "high": 2}

        for text in texts:
            r = extract_demographics(text)
            # Keep highest-confidence age
            if r["age"] is not None and conf_rank.get(r["confidence"], 0) >= conf_rank.get(best["confidence"], 0):
                best["age"] = r["age"]
                best["confidence"] = r["confidence"]
                best["matched_excerpt"] = r["matched_excerpt"]
                best["pattern_type"] = r["pattern_type"]
            if r["age_min"] is not None and best["age"] is None:
                best["age_min"] = r["age_min"]
                best["age_max"] = r["age_max"]
            # Fill gender / education if missing
            if r["gender"] is not None and best["gender"] is None:
                best["gender"] = r["gender"]
            if r["education_level"] is not None and best["education_level"] is None:
                best["education_level"] = r["education_level"]

        return best

    # ============================================================
    # LOAD, PARSE TID, GROUP BY USER, ANNOTATE, INFLATE BACK
    # ============================================================

    df = pd.read_csv(INPUT_CSV)
    print(df.columns)
    # Parse TID
    # parts = df['TID'].str.split('|')
    # df['user_id'] = parts.str[0].astype(int)
    df['post_id'] = df['post_id']
    df['condition'] = df['label']
    df['text'] = df['text']
    print(f"Loaded {len(df)} posts from {df['user_id'].nunique()} users")

    # Group by user_id and run extraction with tqdm
    user_groups = df.groupby('user_id')['text'].apply(list)

    user_demos = {}
    for uid, texts in tqdm(user_groups.items(), total=len(user_groups), desc="Users", file=__import__('sys').stdout):
        user_demos[uid] = extract_user_demographics(texts)

    user_demo_df = pd.DataFrame.from_dict(user_demos, orient='index')
    user_demo_df.index.name = 'user_id'
    user_demo_df = user_demo_df.reset_index()

    # Merge back to post-level (inflate)
    out = df.merge(user_demo_df, on='user_id', how='left')

    # Reorder columns
    out = out[['user_id', 'post_id', 'condition', 'text', 
            'age', 'age_min', 'age_max', 'gender', 'education_level',
            'matched_excerpt', 'pattern_type', 'confidence']]

    out.to_csv(OUTPUT_CSV, index=False)

    # Stats
    print(f"\nDone. {len(out)} posts, {out['user_id'].nunique()} users")
    print(f"Age found:       {(out.groupby('user_id')['age'].first().notna() | out.groupby('user_id')['age_min'].first().notna()).sum()} users")
    print(f"Gender found:    {out.groupby('user_id')['gender'].first().notna().sum()} users")
    print(f"Education found: {out.groupby('user_id')['education_level'].first().notna().sum()} users")
    print(f"Saved to: {OUTPUT_CSV}")

    out.head(10)

In [3]:
# combine csvs into one
files = ['final_ocd','final_autism', 'final_depression', 'final_anxiety', 'final_bipolar'
         'final_ptsd']

# df = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
for f in tqdm(files, desc="Files", file=__import__('sys').stdout):
    run_pattern_extraction(f)

Files:   0%|          | 0/5 [00:00<?, ?it/s]Index(['user_id', 'post_id', 'label', 'text', 'language'], dtype='str')
Loaded 371034 posts from 2081 users
Users: 100%|██████████| 2081/2081 [01:26<00:00, 23.96it/s]

Done. 371034 posts, 2081 users
Age found:       1282 users
Gender found:    556 users
Education found: 1372 users
Saved to: data/demographics/final_ocd.csv
Files:  20%|██        | 1/5 [01:33<06:14, 93.68s/it]Index(['user_id', 'post_id', 'label', 'text', 'language'], dtype='str')
Loaded 1836666 posts from 8455 users
Users: 100%|██████████| 8455/8455 [07:45<00:00, 18.17it/s]

Done. 1836666 posts, 8455 users
Age found:       5551 users
Gender found:    2309 users
Education found: 6103 users
Saved to: data/demographics/final_autism.csv
Files:  40%|████      | 2/5 [09:53<14:50, 296.69s/it]


FileNotFoundError: [Errno 2] No such file or directory: 'data/mindset/final_depression.csv'

In [ ]:
import pandas as pd
df = pd.read_csv('mindset-strong-gender-annot.csv')
# sample 100 random posts
out = df.sample(100, random_state=42)
sample.to_csv('mindset-strong-gender-annot-sample.csv', index=False)

In [4]:
files = ['final_autism', 'final_ocd']
out = pd.concat([pd.read_csv(f"data/demographics/{f}.csv") for f in files], ignore_index=True)
out[out['gender'].notnull()].sample(100, random_state=42)[['text', 'gender', 'matched_excerpt', 'confidence']].to_csv('sample_demographics.csv', index=False)

In [ ]:
# Keep only high-confidence age+gender disclosures (e.g. "24m", "23f", "(26f)")
# and filter to male/female only
df=out
strong = df[(df['confidence'] == 'high') & (df['gender'].isin(['male', 'female']))].copy()

# parse user_id from TID 
strong['user_id'] = strong['TID'].str.split('|').str[0].astype(int)
# ensure unique user ids
strong = strong.drop_duplicates(subset=['user_id'])
# Get unique users per gender
males   = strong[strong['gender'] == 'male']['user_id'].unique()
females = strong[strong['gender'] == 'female']['user_id'].unique()
print(f"Males: {len(males)} users, Females: {len(females)} users")

# Balance: downsample the larger group to match the smaller
n = min(len(males), len(females))
males_sample   = pd.Series(males).sample(n, random_state=42).values
females_sample = pd.Series(females).sample(n, random_state=42).values

balanced = strong[strong['user_id'].isin(set(males_sample) | set(females_sample))]
print(f"Balanced: {n} users per gender, {len(balanced)} total posts")
print(balanced['gender'].value_counts())
balanced.head()

Males: 26293 users, Females: 16100 users
Balanced: 16100 users per gender, 32200 total posts
gender
female    16100
male      16100
Name: count, dtype: int64


,TID,text,created_utc,age,age_min,age_max,gender,education_level,matched_excerpt,pattern_type,confidence,user_id
99,822003|adhd|s22moj,stupid 1st world stuff here but putting it for...,1.641981e+09,32.0,NaN,NaN,female,NaN,32f,age_gender,high,822003
441,791323|adhd|nbijqe,i am 28f but was born with or developed severa...,1.620918e+09,28.0,NaN,NaN,female,NaN,28f,age_gender,high,791323
765,318905|adhd|glrhhf,long post. . . tl dr is my sis wasn't there fo...,1.589764e+09,34.0,NaN,NaN,female,NaN,34f,age_gender,high,318905
1813,516112|adhd|kezfv4e,34 f here. never wanted kids. there was a peri...,1.703598e+09,34.0,NaN,NaN,female,NaN,34 f,age_gender,high,516112
2999,692720|adhd|h7dzak3,thank you sooo much this looks great! ! and on...,1.627874e+09,32.0,NaN,NaN,female,NaN,32f,age_gender,high,692720
